In [13]:
import yfinance as yf
import pandas as pd
import os

# ============================================
# SETTINGS
# ============================================
COINS      = ["BTC-USD", "ETH-USD"]
START_DATE = "2017-01-01"
END_DATE   = "2026-02-12"
RAW_DATA_PATH = "../data/raw"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

all_data = []

# ============================================
# DOWNLOAD EACH COIN
# ============================================
for ticker in COINS:
    print(f"\n📥 Downloading {ticker}...")
    
    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        auto_adjust=True,   # Adjust for splits/dividends
        progress=False
    )
    
    if df.empty:
        print(f"⚠️ Warning: {ticker} returned empty data!")
        continue
    
    # ── Fix multi-level columns ──
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # ── Reset index ──
    df = df.reset_index()
    
    # ── Rename columns ──
    df = df.rename(columns={
        "Date":   "timestamp",
        "Close":  "price",
        "Volume": "volume",
    })
    
    # ── Keep relevant columns ──
    df = df[["timestamp", "price","volume"]]
    
    # ── Add coin name ──
    df["coin"] = ticker.split("-")[0].lower()
    
    print(f"✅ {ticker}: {len(df)} rows downloaded")
    print(f"   Date range: {df['timestamp'].min().date()} → "
          f"{df['timestamp'].max().date()}")
    
    all_data.append(df)

# ============================================
# COMBINE
# ============================================
combined_df = pd.concat(all_data, ignore_index=True)
combined_df = combined_df.sort_values(
    ["coin", "timestamp"]
).reset_index(drop=True)
combined_df = combined_df.dropna().reset_index(drop=True)

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*50)
print("✅ DATASET SUMMARY")
print("="*50)
print(f"Shape: {combined_df.shape}")
print(f"\nRows per coin:")
print(combined_df["coin"].value_counts())
print(f"\nDate range:")
print(f"   Min: {combined_df['timestamp'].min().date()}")
print(f"   Max: {combined_df['timestamp'].max().date()}")
print(f"\nColumns: {combined_df.columns.tolist()}")
print(f"\nSample:")
print(combined_df.head())

# ── Save ──
file_path = os.path.join(RAW_DATA_PATH, "crypto_yfinance_raw.csv")
combined_df.to_csv(file_path, index=False)
print(f"\n✅ Saved to: {file_path}")


📥 Downloading BTC-USD...
✅ BTC-USD: 3328 rows downloaded
   Date range: 2017-01-01 → 2026-02-10

📥 Downloading ETH-USD...
✅ ETH-USD: 3016 rows downloaded
   Date range: 2017-11-09 → 2026-02-10

✅ DATASET SUMMARY
Shape: (6344, 4)

Rows per coin:
coin
btc    3328
eth    3016
Name: count, dtype: int64

Date range:
   Min: 2017-01-01
   Max: 2026-02-10

Columns: ['timestamp', 'price', 'volume', 'coin']

Sample:
Price  timestamp        price     volume coin
0     2017-01-01   998.325012  147775008  btc
1     2017-01-02  1021.750000  222184992  btc
2     2017-01-03  1043.839966  185168000  btc
3     2017-01-04  1154.729980  344945984  btc
4     2017-01-05  1013.380005  510199008  btc

✅ Saved to: ../data/raw\crypto_yfinance_raw.csv


In [11]:
# ============================================
# NaN CHECK
# ============================================

print("="*50)
print("NaN CHECK")
print("="*50)

# Overall NaNs
print(f"\n📊 Overall NaNs per column:")
print(combined_df.isna().sum())

# Per coin
print(f"\n📊 NaNs per coin:")
for coin in ["btc", "eth"]:
    coin_df = combined_df[combined_df["coin"] == coin]
    total_nans = coin_df.isna().sum().sum()
    print(f"\n   {coin.upper()}:")
    print(f"   Total rows: {len(coin_df)}")
    print(f"   Total NaNs: {total_nans}")
    print(coin_df.isna().sum())

# Date continuity check
print(f"\n📊 Date continuity check:")
for coin in ["btc", "eth"]:
    coin_df = combined_df[combined_df["coin"] == coin].copy()
    coin_df["timestamp"] = pd.to_datetime(coin_df["timestamp"])
    coin_df = coin_df.sort_values("timestamp")
    
    # Check gaps
    diffs = coin_df["timestamp"].diff().dropna()
    gaps  = diffs[diffs > pd.Timedelta(days=1)]
    
    print(f"\n   {coin.upper()}:")
    print(f"   Expected days: {(coin_df['timestamp'].max() - coin_df['timestamp'].min()).days}")
    print(f"   Actual rows:   {len(coin_df)}")
    print(f"   Gaps > 1 day:  {len(gaps)}")
    if len(gaps) > 0:
        print(f"   Largest gap:   {gaps.max()}")

NaN CHECK

📊 Overall NaNs per column:
Price
timestamp    0
price        0
open         0
high         0
low          0
volume       0
coin         0
dtype: int64

📊 NaNs per coin:

   BTC:
   Total rows: 3328
   Total NaNs: 0
Price
timestamp    0
price        0
open         0
high         0
low          0
volume       0
coin         0
dtype: int64

   ETH:
   Total rows: 3016
   Total NaNs: 0
Price
timestamp    0
price        0
open         0
high         0
low          0
volume       0
coin         0
dtype: int64

📊 Date continuity check:

   BTC:
   Expected days: 3327
   Actual rows:   3328
   Gaps > 1 day:  0

   ETH:
   Expected days: 3015
   Actual rows:   3016
   Gaps > 1 day:  0
